# Time-Series Baseline Selection and Evidence Review

This section fixes the two formal time-series baselines for the frozen primary task and records their provenance in `baseline_reproduction_spec`: GlucoBench Linear Regression (primary) and the Sparacino et al. (2007) time-varying AR(1) model with exponentially weighted least squares (secondary). Every specification entry is labelled as paper-reported, public-code-verified, not reported, project adaptation, or project-specified decision. No model is fitted and no prediction or metric is computed here.

In [1]:
# --- Import dependencies and configure artifact paths ---

import json
import re
import time
import os

# Fix the physical-core count so joblib/loky does not emit a non-actionable Windows detection warning.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from src.paths import PROJECT_ROOT

EDA_DIR = PROJECT_ROOT / "artifacts" / "03_eda_task_samples"
BASELINE_DIR = PROJECT_ROOT / "artifacts" / "04_baseline_models"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 90)

# --- Define the error-summary helper ---

def error_summary(errors: np.ndarray) -> dict:
    return {"mae": float(np.mean(np.abs(errors))), "rmse": float(np.sqrt(np.mean(errors ** 2))), "windows": int(len(errors))}

In [2]:
# Frozen Notebook 03 artifacts that define the prediction task, samples, splits and features (read-only).
UPSTREAM_FILES = {
    "task_protocol": EDA_DIR / "task_protocol.json",
    "window_construction_rules": EDA_DIR / "window_construction_rules.json",
    "split_rules": EDA_DIR / "split_rules.json",
    "feature_registry": EDA_DIR / "feature_registry.csv",
    "sample_flow": EDA_DIR / "sample_flow.csv",
    "window_manifest": EDA_DIR / "window_manifest.parquet",
    "fold_assignments": EDA_DIR / "fold_assignments.parquet",
    "glucose_history_samples": EDA_DIR / "glucose_history_samples.parquet",
}

task_protocol = json.loads(UPSTREAM_FILES["task_protocol"].read_text(encoding="utf-8"))
window_rules = json.loads(UPSTREAM_FILES["window_construction_rules"].read_text(encoding="utf-8"))
split_rules = json.loads(UPSTREAM_FILES["split_rules"].read_text(encoding="utf-8"))
feature_registry = pd.read_csv(UPSTREAM_FILES["feature_registry"])
sample_flow = pd.read_csv(UPSTREAM_FILES["sample_flow"])
primary_task = task_protocol["primary_task"]


canonical_folds = pd.read_parquet(UPSTREAM_FILES["fold_assignments"])
canonical_samples = pd.read_parquet(UPSTREAM_FILES["glucose_history_samples"])
folds = canonical_folds[["task_id", "subject", "origin_time", "evaluation_protocol", "split", "admitted_to_split"]].copy()
samples = canonical_samples[["task_id", "subject", "origin_time"]].copy()

WINDOW_KEY = split_rules["window_key"]
FOLD_KEY = split_rules["fold_key"]
eligible_windows = int(sample_flow.loc[sample_flow["rule"].eq("eligible_windows"), "remaining_count"].iloc[0])
assert len(samples) == eligible_windows
assert not samples.duplicated(WINDOW_KEY).any()
assert not folds.duplicated(FOLD_KEY).any()

# Evaluation protocols and admitted split sizes.
PROTOCOL_A = "Within-subject Future Prediction"
PROTOCOL_B = "Unseen-subject Generalization"
assert set(folds["evaluation_protocol"]) == {PROTOCOL_A, PROTOCOL_B}
split_sizes = (
    folds.loc[folds["admitted_to_split"]]
    .groupby(["evaluation_protocol", "split"], sort=True)
    .agg(windows=("subject", "size"), subjects=("subject", "nunique"))
    .reset_index()
)
registered_features = feature_registry.loc[feature_registry["allowed_as_predictor"], "feature_name"].tolist()

frozen_task = {
    "protocol_id": task_protocol["protocol_id"],
    "task_id": primary_task["task_id"],
    "target_device": primary_task["target_device"],
    "target_field": primary_task["target_field"],
    "target_unit": primary_task["target_unit"],
    "prediction_type": primary_task["prediction_type"],
    "history_interval": primary_task["history_interval"],
    "history_minutes": primary_task["history_minutes"],
    "expected_history_points": window_rules["expected_history_points"],
    "forecast_horizon_minutes": primary_task["forecast_horizon_minutes"],
    "origin_stride_minutes": primary_task["origin_stride_minutes"],
    "grid_resolution_minutes": primary_task["grid_resolution_minutes"],
    "base_grid_artifact": primary_task["base_grid_artifact"],
    "input_cutoff_rule": task_protocol["input_availability_policy"]["global_cutoff_rule"],
    "window_rule_version": window_rules["rule_version"],
    "window_key": WINDOW_KEY,
    "fold_key": FOLD_KEY,
    "split_rule_version": split_rules["version"],
    "evaluation_protocols": [PROTOCOL_A, PROTOCOL_B],
    "temporal_partitions": split_rules["temporal_partitions"],
    "subject_rule": split_rules["subject_rule"],
    "support_rule": split_rules["support_rule"],
    "preprocessing_policy": split_rules["preprocessing_policy"],
    "unseen_history_policy": split_rules["unseen_history_policy"],
    "cross_protocol_policy": split_rules["cross_protocol_policy"],
    "eligible_windows": eligible_windows,
    "registered_glucose_features": registered_features,
}

print(f"eligible windows: {eligible_windows}")
display(split_sizes)

eligible windows: 134539


,evaluation_protocol,split,windows,subjects
0,Unseen-subject Generalization,test,28322,9
1,Unseen-subject Generalization,train,81958,27
2,Unseen-subject Generalization,validation,24259,9
3,Within-subject Future Prediction,test,26269,45
4,Within-subject Future Prediction,train,79300,45
5,Within-subject Future Prediction,validation,26330,45


In [3]:
# Provenance vocabulary for every specification entry.
PAPER = "paper-reported"
CODE = "public-code-verified"
NOT_REPORTED = "not reported"
ADAPTED = "project adaptation"
DECISION = "project-specified decision"
PROVENANCE = [PAPER, CODE, NOT_REPORTED, ADAPTED, DECISION]

GLUCOBENCH_COMMIT = "661d840a98b316df51faa13a7100430afcbbb5b7"

# Bibliographic and implementation sources behind the two formal baselines.
sources = pd.DataFrame([
    {"source_id": "glucobench_paper", "evidence_role": "primary",
     "citation": ("Sergazinov R, Chun E, Rogovchenko V, Fernandes N, Kasman N, Gaynanova I. GlucoBench: Curated "
                  "List of Continuous Glucose Monitoring Datasets with Prediction Benchmarks. ICLR 2024."),
     "identifier": "arXiv:2410.05780v1", "version": "v1", "access": "full text", "access_date": "2026-09-27"},
    {"source_id": "glucobench_code", "evidence_role": "primary",
     "citation": "IrinaStatsLab/GlucoBench official repository (https://github.com/IrinaStatsLab/GlucoBench)",
     "identifier": f"commit {GLUCOBENCH_COMMIT}",
     "version": "lib/linreg.py; utils/darts_processing.py; data_formatter/base.py; config/*.yaml; requirements.txt",
     "access": "source files read", "access_date": "2026-09-27"},
    {"source_id": "darts_docs", "evidence_role": "primary",
     "citation": "Darts documentation, darts.models.forecasting.linear_regression_model.LinearRegressionModel",
     "identifier": "unit8co.github.io/darts", "version": "online documentation (repository pin darts==0.29.0)",
     "access": "documentation read", "access_date": "2026-09-27"},
    {"source_id": "sparacino2007", "evidence_role": "primary",
     "citation": ("Sparacino G, Zanderigo F, Corazza S, Maran A, Facchinetti A, Cobelli C. Glucose concentration "
                  "can be predicted ahead in time from continuous glucose monitoring sensor time-series. "
                  "IEEE Trans Biomed Eng. 2007;54(5):931-937."),
     "identifier": "doi:10.1109/TBME.2006.889774", "version": "published article",
     "access": "full text", "access_date": "2026-09-28"},
    {"source_id": "zanderigo2007", "evidence_role": "secondary",
     "citation": ("Zanderigo F, Sparacino G, Kovatchev B, Cobelli C. Glucose prediction algorithms from continuous "
                  "monitoring data. J Diabetes Sci Technol. 2007;1(5):645-651."),
     "identifier": "PMC2734107", "version": "published article", "access": "full text", "access_date": "2026-09-27"},
    {"source_id": "cobelli2009", "evidence_role": "secondary",
     "citation": ("Cobelli C, Dalla Man C, Sparacino G, Magni L, De Nicolao G, Kovatchev BP. Diabetes: models, "
                  "signals, and control. IEEE Rev Biomed Eng. 2009;2:54-96."),
     "identifier": "PMC2951686", "version": "published article", "access": "full text", "access_date": "2026-09-27"},
])
assert sources["source_id"].is_unique
display(sources[["source_id", "evidence_role", "identifier", "access"]])

,source_id,evidence_role,identifier,access
0,glucobench_paper,primary,arXiv:2410.05780v1,full text
1,glucobench_code,primary,commit 661d840a98b316df51faa13a7100430afcbbb5b7,source files read
2,darts_docs,primary,unit8co.github.io/darts,documentation read
3,sparacino2007,primary,doi:10.1109/TBME.2006.889774,full text
4,zanderigo2007,secondary,PMC2734107,full text
5,cobelli2009,secondary,PMC2951686,full text


In [4]:
# Define frozen task geometry and baseline specifications.
HISTORY_MIN = frozen_task["history_minutes"]
HORIZON_MIN = frozen_task["forecast_horizon_minutes"]
GRID_MIN = frozen_task["grid_resolution_minutes"]
TARGET_FIELD = frozen_task["target_field"]

# Sparacino geometry: the published method uses a 3-min sampling interval and T = 10 steps for PH = 30 min.
SPARACINO_STEP_MIN = 3
SPARACINO_PAPER_T_FOR_PH30 = 10
sparacino_offsets = list(range(-HISTORY_MIN, 1, SPARACINO_STEP_MIN))
sparacino_geometry = {
    "input_window": frozen_task["history_interval"],
    "step_minutes": SPARACINO_STEP_MIN,
    "observation_offsets_minutes": sparacino_offsets,
    "observations": len(sparacino_offsets),
    "regression_pairs": len(sparacino_offsets) - 1,
    "forecast_steps_T": HORIZON_MIN // SPARACINO_STEP_MIN,
    "output": f"single {frozen_task['target_field']} prediction at t+{HORIZON_MIN} min",
}


def entry(field, value, provenance, source_ids=""):
    return {"field": field, "value": value, "provenance": provenance, "source_ids": source_ids}


# Primary baseline: GlucoBench linear regression.
linreg_fields = [
    entry("model_name", "GlucoBench Linear Regression (global linear autoregressive model on target lags)", PAPER, "glucobench_paper"),
    entry("source_paper", "GlucoBench, ICLR 2024 (arXiv:2410.05780v1), Section 4.2", PAPER, "glucobench_paper"),
    entry("implementation_source", f"lib/linreg.py @ {GLUCOBENCH_COMMIT}: darts.models.LinearRegressionModel(lags=in_len, lags_past_covariates=None, lags_future_covariates=None, output_chunk_length=out_len)", CODE, "glucobench_code"),
    entry("original_task", "1-hour-ahead CGM forecasting, T = 12 steps at 5-min frequency", PAPER, "glucobench_paper"),
    entry("original_input_geometry", "Past target values only; lags = in_len points; in_len searched by Optuna in [12, max_length_input] with step 12 (saved configs: iglu 192, hall 84)", CODE, "glucobench_code"),
    entry("original_input_geometry", "load_data(..., use_static_covs=True) attaches static covariates (including the subject id) to target series", CODE, "glucobench_code"),
    entry("original_fitting", "Separate linear model for each forecast step", PAPER, "glucobench_paper"),
    entry("original_fitting", "Default estimator sklearn.linear_model.LinearRegression (ordinary least squares); multi_models=True trains one model per step; fitted once on all training series", CODE, "darts_docs;glucobench_code"),
    entry("original_fitting", "Min-max scaling fitted on the training target across all series (ScalerCustom); short gaps linearly interpolated", CODE, "glucobench_code;glucobench_paper"),
    entry("original_horizon", "12 steps x 5 min = 60 min trajectory", PAPER, "glucobench_paper"),
    entry("current_role", "primary formal time-series baseline", DECISION),
    entry("project_adaptation", f"Single direct prediction of {frozen_task['target_field']} at t+{HORIZON_MIN} min, i.e. only the {HORIZON_MIN}-min step of the per-step strategy", ADAPTED),
    entry("project_adaptation", f"History fixed to the frozen {HISTORY_MIN}-min window; no input-length search", ADAPTED),
    entry("project_adaptation", "Training rows are the frozen Notebook 03 training windows; one pooled model per evaluation protocol", ADAPTED),
    entry("project_adaptation", "No static covariates and no subject identifier as regressors (subject identity is a grouping key only)", ADAPTED),
    entry("project_adaptation", "Any scaling is fitted on the training split of the same protocol only; no gap interpolation (strict published windows)", ADAPTED),
    entry("unresolved_detail", "Estimator keyword arguments (e.g. fit_intercept) are not set in the reference code; the value must be set explicitly", NOT_REPORTED, "glucobench_code"),
    entry("unresolved_detail", "Whether darts 0.29.0 uses the attached static covariates as regressors was not verified", NOT_REPORTED, "glucobench_code"),
    entry("implementation_status", "Reference implementation read at the pinned commit; implemented and fitted in the training section below", CODE, "glucobench_code"),
]

# Secondary baseline: Sparacino et al. (2007) time-varying AR(1) with exponentially weighted least squares.
sparacino_fields = [
    entry("model_name", "Sparacino et al. (2007) first-order autoregressive model with time-varying parameters (AR(1)-WLS)", PAPER, "sparacino2007"),
    entry("source_paper", "IEEE Trans Biomed Eng 2007;54(5):931-937, full text", PAPER, "sparacino2007"),
    entry("implementation_source", "No author, same-group or third-party reference implementation available", NOT_REPORTED, "sparacino2007"),
    entry("original_task", "Ahead-of-time CGM glucose prediction with PH = 30 and 45 min", PAPER, "sparacino2007"),
    entry("original_input_geometry", "All past samples of the same subject, 3-min GlucoDay series; sample k instants before t_n weighted by mu^k, mu in (0, 1)", PAPER, "sparacino2007"),
    entry("original_model", "u_i = a * u_(i-1) + w_i; theta = (a, sigma^2); no intercept", PAPER, "sparacino2007"),
    entry("original_fitting", "theta re-estimated at every sampling time by weighted least squares; AR model solved with the recursive least squares algorithm of the Matlab System Identification Toolbox", PAPER, "sparacino2007"),
    entry("original_prediction", "Equation iterated for i = n+1 ... n+T with w_i = 0; T = 10 or 15 at 3 min gives PH 30 or 45 min", PAPER, "sparacino2007"),
    entry("original_hyperparameters", "mu = 0.2, 0.5, 0.8 compared; no single value selected; mu = 1 rejected", PAPER, "sparacino2007"),
    entry("original_preprocessing", "First-order Butterworth low-pass filter, cutoff normalized to half the sampling rate = 0.05", PAPER, "sparacino2007"),
    entry("original_data", "28 type 1 diabetic patients, GlucoDay microdialysis CGM, 24-48 h home monitoring", PAPER, "sparacino2007"),
    entry("original_evaluation", "MSPE and ESOD; delay of threshold crossings; each subject processed on its own series", PAPER, "sparacino2007"),
    entry("secondary_consistency", "Same equation and exponential weighting described by the same group", PAPER, "zanderigo2007;cobelli2009"),
    entry("current_role", "secondary formal time-series baseline", DECISION),
    entry("reproduction_wording", "method reproduction based on published equations; no reference implementation available", DECISION),
    entry("project_adaptation", f"Input restricted to the frozen window {frozen_task['history_interval']} on the released {GRID_MIN}-min Libre grid", ADAPTED),
    entry("project_adaptation", f"Observations every {SPARACINO_STEP_MIN} min at offsets {sparacino_offsets[0]} ... {sparacino_offsets[-1]} min: {sparacino_geometry['observations']} observations, {sparacino_geometry['regression_pairs']} regression pairs", ADAPTED),
    entry("project_adaptation", f"Iterative forecast with innovation set to zero for T = {sparacino_geometry['forecast_steps_T']} steps; output is the single t+{HORIZON_MIN} min value", ADAPTED),
    entry("project_adaptation", "Coefficient a estimated independently in every window; no state carried across windows", ADAPTED),
    entry("unresolved_detail", "RLS toolbox function and initialization, filter causality, start-up length and missing-data handling are not reported; resolved for this project by decisions R1 and R3 and by complete frozen windows", NOT_REPORTED, "sparacino2007"),
    entry("implementation_status", "published-equation method reproduction; reference implementation unavailable; implemented and fitted in the training section below", DECISION),
]

baseline_fields = pd.concat([
    pd.DataFrame(linreg_fields).assign(model_id="glucobench_linear_regression"),
    pd.DataFrame(sparacino_fields).assign(model_id="sparacino2007_ar1_wls"),
], ignore_index=True)

# Provenance checks: controlled vocabulary, cited sources exist, adaptations and decisions carry no literature citation.
assert baseline_fields["provenance"].isin(PROVENANCE).all()
cited = baseline_fields["source_ids"].str.split(";").explode()
assert set(cited[cited.ne("")]) <= set(sources["source_id"])
assert baseline_fields.loc[baseline_fields["provenance"].isin([ADAPTED, DECISION]), "source_ids"].eq("").all()
assert not baseline_fields.loc[baseline_fields["model_id"].eq("sparacino2007_ar1_wls"), "provenance"].eq(CODE).any()

print(f"Sparacino geometry: {sparacino_geometry['observations']} observations, "
      f"{sparacino_geometry['regression_pairs']} pairs, T = {sparacino_geometry['forecast_steps_T']}")
display(baseline_fields.pivot_table(index="model_id", columns="provenance", values="field",
                                    aggfunc="count", fill_value=0).reindex(columns=PROVENANCE, fill_value=0))

Sparacino geometry: 41 observations, 40 pairs, T = 10


provenance,paper-reported,public-code-verified,not reported,project adaptation,project-specified decision
model_id,,,,,
glucobench_linear_regression,5,6,2,5,1
sparacino2007_ar1_wls,12,0,2,4,3


The Sparacino baseline is a method reproduction based on published equations; no reference implementation is available. Its window geometry (3-min observations, T = 10) follows from the frozen history and horizon.

In [5]:
# Implementation decisions for the Sparacino baseline.
sparacino_decisions = [
    {"decision_id": "R1", "topic": "filtering", "provenance": DECISION,
     "rule": "Main analysis uses no low-pass filter. The published first-order Butterworth filter may only be applied causally in an optional sensitivity analysis; zero-phase filtering is not permitted.",
     "classification": ADAPTED},
    {"decision_id": "R2", "topic": "forgetting factor", "provenance": DECISION,
     "rule": "Candidate set mu in {0.2, 0.5, 0.8}. The formal mu is selected on validation data only, separately within each evaluation protocol, and locked before test evaluation. Test data never inform mu. Results for all three values may be reported as a pre-specified sensitivity.",
     "classification": ADAPTED},
    {"decision_id": "R3", "topic": "estimator", "provenance": DECISION,
     "rule": "Closed-form implementation of the published exponentially weighted AR(1) estimator: a = sum_k mu^k u(n-k) u(n-k-1) / sum_k mu^k u(n-k-1)^2 over the window pairs; forecast a^T * u(n). No claim of numerical identity with the original Matlab RLS.",
     "classification": ADAPTED},
]

# Evaluation-protocol rules applied to both baselines.
protocol_rules = {
    PROTOCOL_A: {
        "rule": "Models and any learned preprocessing are fitted on each subject's training partition only; validation guides model and hyperparameter selection; test is used only for frozen evaluation.",
        "provenance": "Notebook 03 split_rules",
    },
    PROTOCOL_B: {
        "rule": ("Test subjects may not contribute to population-level fitting, reusable subject-specific fitting, "
                 "preprocessing fitting, or hyperparameter tuning. Deterministic calculations based solely on the "
                 "current causally available input window are permitted at prediction time."),
        "prohibited": ["population-level model fitting on test subjects",
                       "reusable subject-specific fitting on test subjects",
                       "preprocessing fitted on test-subject statistics",
                       "hyperparameter tuning on test subjects",
                       "information after the prediction origin",
                       "test labels"],
        "permitted": ["deterministic calculation on the current window [t-120, t] only",
                      "local feature calculation",
                      "local closed-form coefficient estimation, including the Sparacino coefficient a"],
        "conditions": ["belongs to the current window only", "not stored across windows",
                       "not reused for future windows", "no future information", "no test labels",
                       "hyperparameters fixed on permitted data"],
        "provenance": DECISION,
    },
}
baseline_protocol_use = pd.DataFrame([
    {"model_id": "glucobench_linear_regression", "protocol": PROTOCOL_A,
     "fitting": "pooled model fitted on the training partitions", "prediction_time": "frozen window lags"},
    {"model_id": "glucobench_linear_regression", "protocol": PROTOCOL_B,
     "fitting": "pooled model fitted on training subjects only", "prediction_time": "frozen window lags"},
    {"model_id": "sparacino2007_ar1_wls", "protocol": PROTOCOL_A,
     "fitting": "none; mu selected on this protocol's validation split",
     "prediction_time": "local a estimated from the current window only"},
    {"model_id": "sparacino2007_ar1_wls", "protocol": PROTOCOL_B,
     "fitting": "none; mu selected on this protocol's validation split",
     "prediction_time": "local a estimated from the current window only (permitted deterministic calculation)"},
])


display(pd.DataFrame(sparacino_decisions)[["decision_id", "topic", "provenance"]])
display(baseline_protocol_use)

,decision_id,topic,provenance
0,R1,filtering,project-specified decision
1,R2,forgetting factor,project-specified decision
2,R3,estimator,project-specified decision


,model_id,protocol,fitting,prediction_time
0,glucobench_linear_regression,Within-subject Future Prediction,pooled model fitted on the training partitions,frozen window lags
1,glucobench_linear_regression,Unseen-subject Generalization,pooled model fitted on training subjects only,frozen window lags
2,sparacino2007_ar1_wls,Within-subject Future Prediction,none; mu selected on this protocol's validation split,local a estimated from the current window only
3,sparacino2007_ar1_wls,Unseen-subject Generalization,none; mu selected on this protocol's validation split,local a estimated from the current window only (permitted deterministic calculation)


In [6]:
# Record the final reproducible baseline specification.
spec_id = "baseline_reproduction_spec_v2"
# Final baseline configuration: GlucoBench lag spacing is 5-min target lags (in_len in {12, 24}); the
# implementation route is a direct sklearn LinearRegression with fit_intercept=True, explicit; both are
# implemented in the training section below. The Sparacino causal-filter sensitivity (R1) is an optional
# sensitivity analysis, not part of the main analysis, and is recorded under sparacino_decisions above.

spec = {
    "spec_id": spec_id,
    "selection_status": "formal baselines selected",
    "provenance_vocabulary": PROVENANCE,
    "frozen_task": frozen_task,
    "upstream_artifacts": {name: path.relative_to(PROJECT_ROOT).as_posix() for name, path in UPSTREAM_FILES.items()},
    "split_sizes": split_sizes.to_dict(orient="records"),
    "sources": sources.to_dict(orient="records"),
    "baselines": {
        "glucobench_linear_regression": {
            "role": "primary",
            "reproduction_status": "public implementation verified / adapted method reproduction",
            "fields": baseline_fields.loc[baseline_fields["model_id"].eq("glucobench_linear_regression"),
                                          ["field", "value", "provenance", "source_ids"]].to_dict(orient="records"),
        },
        "sparacino2007_ar1_wls": {
            "role": "secondary",
            "reproduction_status": "published-equation method reproduction / reference implementation unavailable",
            "geometry": sparacino_geometry,
            "decisions": sparacino_decisions,
            "fields": baseline_fields.loc[baseline_fields["model_id"].eq("sparacino2007_ar1_wls"),
                                          ["field", "value", "provenance", "source_ids"]].to_dict(orient="records"),
        },
    },
    "protocol_rules": protocol_rules,
    "baseline_protocol_use": baseline_protocol_use.to_dict(orient="records"),
}

frozen_spec = spec

# Flat view: one row per specification entry.
flat = [baseline_fields.assign(record_type="baseline_field")]
flat.append(pd.DataFrame(sparacino_decisions).assign(
    record_type="decision", model_id="sparacino2007_ar1_wls", field=lambda d: d["decision_id"] + " " + d["topic"],
    value=lambda d: d["rule"], source_ids=""))
flat.append(pd.DataFrame([{"record_type": "protocol_rule", "model_id": "all", "field": name,
                           "value": rule["rule"], "provenance": rule["provenance"], "source_ids": ""}
                          for name, rule in protocol_rules.items()]))
spec_table = pd.concat(flat, ignore_index=True)[["record_type", "model_id", "field", "value", "provenance", "source_ids"]]
spec_table.insert(0, "spec_id", spec_id)

json_path = BASELINE_DIR / "baseline_reproduction_spec.json"
json_path.write_text(json.dumps(spec, indent=2, ensure_ascii=True) + "\n", encoding="utf-8")
assert spec_table["provenance"].isin(PROVENANCE + ["Notebook 03 split_rules"]).all()

Sparacino decisions: no filtering in the main analysis (R1); $\mu$ selected from $\{0.2, 0.5, 0.8\}$ on validation data within each protocol (R2); closed-form exponentially weighted estimator cross-checked against an independent RLS implementation (R3). Under Protocol B, test subjects contribute no fitting, preprocessing or tuning; window-only deterministic calculations, including the local Sparacino coefficient, are permitted.

# Persistence and Trend References

Two simple references are computed from glucose history only: persistence, $\hat{y}(t+30) = G(t)$, and linear trend extrapolation, $\hat{y}(t+30) = G(t) + 30 \times \text{glucose\_slope\_30min}$ (registered trailing OLS slope over $[t-30, t]$; single pre-specified variant, not clipped). Neither fits any parameter, and neither is a formal baseline reproduction. Predictions are produced for the validation windows of both protocols; MAE and RMSE are computed per subject and averaged with equal subject weight, with pooled-window metrics reported alongside.

In [7]:
# Load the frozen task and validation samples for reference predictions.
spec_task = frozen_spec["frozen_task"]
spec_upstream = frozen_spec["upstream_artifacts"]
TREND_FEATURE = "glucose_slope_30min"
CURRENT_FEATURE = "glucose_lag_0min"
feature_registry_by_name = feature_registry.set_index("feature_name")
assert feature_registry_by_name.loc[CURRENT_FEATURE, "lag_minutes"] == 0 and feature_registry_by_name.loc[CURRENT_FEATURE, "source_signal"] == TARGET_FIELD
assert feature_registry_by_name.loc[TREND_FEATURE, "maximum_source_time_relative_to_origin_min"] == 0
assert feature_registry_by_name.loc[[CURRENT_FEATURE, TREND_FEATURE], "allowed_as_predictor"].all()

# Validation windows of both protocols; training and test rows are dropped before any other data are joined.
fold_columns = FOLD_KEY + ["split", "admitted_to_split", "history_start_time", "target_time",
                           "partition_start_time", "partition_end_time", "partition_start_inclusive"]
folds = canonical_folds[fold_columns].copy()
validation = folds.loc[folds["admitted_to_split"] & folds["split"].eq("validation")].drop(columns="admitted_to_split")
assert set(validation["evaluation_protocol"]) == set(spec_task["evaluation_protocols"])

sample_columns = WINDOW_KEY + ["target_time", "target_field", "target_value", "predictor_min_source_time",
                               "predictor_max_source_time", CURRENT_FEATURE, TREND_FEATURE]
samples = canonical_samples[sample_columns].copy()
validation = validation.merge(samples, on=WINDOW_KEY, how="left", suffixes=("", "_sample"), validate="many_to_one")
del samples
assert validation["target_value"].notna().all() and validation["target_time"].eq(validation["target_time_sample"]).all()
validation = validation.drop(columns="target_time_sample").sort_values(FOLD_KEY).reset_index(drop=True)

print(f"Frozen spec: {frozen_spec['spec_id']}; horizon {HORIZON_MIN} min; target {TARGET_FIELD}")
display(validation.groupby("evaluation_protocol").agg(windows=("subject", "size"), subjects=("subject", "nunique")))

Frozen spec: baseline_reproduction_spec_v2; horizon 30 min; target libre_gl


,windows,subjects
evaluation_protocol,,
Unseen-subject Generalization,24259,9
Within-subject Future Prediction,26330,45


In [8]:
# Simple references computed from glucose history available at the origin only; nothing is fitted.
REFERENCES = {
    "persistence": "G(t): published glucose at the prediction origin",
    "linear_trend_30min": f"G(t) + {HORIZON_MIN} min x {TREND_FEATURE} (OLS slope over [t-30, t])",
}
predictions = pd.concat([
    validation.assign(reference="persistence", prediction=validation[CURRENT_FEATURE]),
    validation.assign(reference="linear_trend_30min",
                      prediction=validation[CURRENT_FEATURE] + HORIZON_MIN * validation[TREND_FEATURE]),
], ignore_index=True)
predictions["error"] = predictions["prediction"] - predictions["target_value"]
assert predictions["prediction"].notna().all()

display(pd.Series(REFERENCES, name="definition").to_frame())
# Trend extrapolation is not bounded; physically impossible values are counted, not clipped.
below_zero = predictions.loc[predictions["prediction"] < 0].groupby(["evaluation_protocol", "reference"]).size()
print(f"Predictions below 0 mg/dL: {int(below_zero.sum())} ({below_zero.to_dict()})")
display(predictions.groupby(["evaluation_protocol", "reference"])["prediction"].describe()[["count", "mean", "min", "max"]])

,definition
persistence,G(t): published glucose at the prediction origin
linear_trend_30min,"G(t) + 30 min x glucose_slope_30min (OLS slope over [t-30, t])"


Predictions below 0 mg/dL: 14 ({('Unseen-subject Generalization', 'linear_trend_30min'): 7, ('Within-subject Future Prediction', 'linear_trend_30min'): 7})


count        mean   min    max
evaluation_protocol              reference                                           
Unseen-subject Generalization    linear_trend_30min  24259.0  121.224471 -16.0  425.0
                                 persistence         24259.0  121.178490  40.0  342.0
Within-subject Future Prediction linear_trend_30min  26330.0  109.476233 -13.0  416.0
                                 persistence         26330.0  109.415987  40.0  356.0

In [9]:
# Independent lookup of origin and target values on the frozen 1-minute grid.
base_grid = pd.read_parquet(PROJECT_ROOT / spec_task["base_grid_artifact"],
                            columns=["subject", "timestamp", TARGET_FIELD, "is_published_source_row"])
grid = base_grid.loc[base_grid["subject"].isin(validation["subject"].unique())]
origin_lookup = grid.rename(columns={"timestamp": "origin_time", TARGET_FIELD: "grid_origin_value",
                                     "is_published_source_row": "origin_published"})
target_lookup = grid.rename(columns={"timestamp": "target_time", TARGET_FIELD: "grid_target_value",
                                     "is_published_source_row": "target_published"})
audit = (validation.merge(origin_lookup, on=["subject", "origin_time"], how="left", validate="many_to_one")
         .merge(target_lookup, on=["subject", "target_time"], how="left", validate="many_to_one"))

# Timing and label checks.
start_ok = np.where(audit["partition_start_inclusive"],
                    audit["history_start_time"] >= audit["partition_start_time"],
                    audit["history_start_time"] > audit["partition_start_time"])
checks = {
    "persistence equals grid value at origin": np.allclose(audit[CURRENT_FEATURE], audit["grid_origin_value"], rtol=0, atol=1e-9),
    "origin value is a published source row": audit["origin_published"].eq(True).all(),
    "target equals grid value at origin + horizon": np.allclose(audit["target_value"], audit["grid_target_value"], rtol=0, atol=1e-9),
    "target value is a published source row": audit["target_published"].eq(True).all(),
    "support starts inside validation partition": bool(start_ok.all()),
    "target inside validation partition": (audit["target_time"] <= audit["partition_end_time"]).all(),
}

# The current value must not act as the label: exact equality is reported, and it cannot hold for all windows.
equal_share = audit[CURRENT_FEATURE].eq(audit["target_value"]).groupby(audit["evaluation_protocol"]).mean()
checks["target is not the origin value for every window"] = bool((equal_share < 1).all())

# Protocol B validation subjects are disjoint from Protocol B training subjects.
protocol_b = spec_task["evaluation_protocols"][1]
b_train_subjects = set(folds.loc[folds["evaluation_protocol"].eq(protocol_b) & folds["split"].eq("train")
                                 & folds["admitted_to_split"], "subject"])
b_val_subjects = set(audit.loc[audit["evaluation_protocol"].eq(protocol_b), "subject"])
checks["Protocol B validation subjects unseen in training"] = b_train_subjects.isdisjoint(b_val_subjects)

# Recompute the 30-min trailing OLS slope for a fixed sample of windows directly from the grid.
unique_windows = audit.drop_duplicates(WINDOW_KEY)
slope_sample = unique_windows.sample(n=min(500, len(unique_windows)), random_state=0)
grid_indexed = grid.set_index(["subject", "timestamp"])[TARGET_FIELD]
slope_diff = []
for row in slope_sample.itertuples(index=False):
    times = pd.date_range(row.origin_time - pd.Timedelta(minutes=30), row.origin_time, freq="1min")
    values = grid_indexed.loc[[(row.subject, ts) for ts in times]].to_numpy()
    slope = np.polyfit(np.arange(-30, 1, dtype=float), values, 1)[0]
    slope_diff.append(abs(slope - getattr(row, TREND_FEATURE)))
checks["30-min slope matches independent recomputation (sample)"] = float(np.max(slope_diff)) < 1e-8

audit_table = pd.Series(checks, name="passed").to_frame()
assert audit_table["passed"].all(), audit_table
print("Share of windows where target equals the origin value:")
display(equal_share.rename("share").to_frame())
print(f"Slope recomputation: {len(slope_sample)} windows, max abs difference {max(slope_diff):.2e}")

Share of windows where target equals the origin value:


,share
evaluation_protocol,
Unseen-subject Generalization,0.023744
Within-subject Future Prediction,0.043790


Slope recomputation: 500 windows, max abs difference 7.11e-15


Origin values and labels were re-read from the aligned 1-minute grid; the independently recomputed 30-min slopes agree with the registered feature to numerical precision.

In [10]:
# Aggregate validation reference metrics by protocol and reference.
group_keys = ["evaluation_protocol", "reference"]
reference_subject_metrics = (predictions.groupby(group_keys + ["subject"])["error"]
                             .apply(lambda errors: pd.Series(error_summary(errors.to_numpy()))).unstack().reset_index())
reference_subject_metrics["windows"] = reference_subject_metrics["windows"].astype(int)

# Primary aggregation: per-subject metrics averaged with equal subject weight; pooled windows reported alongside.
summary_rows = []
for (protocol, reference), part in predictions.groupby(group_keys):
    per_subject = reference_subject_metrics.loc[reference_subject_metrics["evaluation_protocol"].eq(protocol)
                                                & reference_subject_metrics["reference"].eq(reference)]
    pooled = error_summary(part["error"].to_numpy())
    summary_rows.append({
        "evaluation_protocol": protocol, "reference": reference, "split": "validation",
        "subjects": len(per_subject), "windows": int(pooled["windows"]),
        "windows_per_subject_min": int(per_subject["windows"].min()),
        "windows_per_subject_median": float(per_subject["windows"].median()),
        "windows_per_subject_max": int(per_subject["windows"].max()),
        "subject_mean_mae": per_subject["mae"].mean(), "subject_mean_rmse": per_subject["rmse"].mean(),
        "pooled_mae": pooled["mae"], "pooled_rmse": pooled["rmse"],
    })
summary_metrics = pd.DataFrame(summary_rows)

# Reference predictions keyed by the fold key; the model comparison below uses them directly.
prediction_columns = FOLD_KEY + ["split", "target_time", "target_field", "target_value", "reference", "prediction", "error"]
reference_predictions = predictions[prediction_columns]
assert not reference_predictions.duplicated(FOLD_KEY + ["reference"]).any()

display(summary_metrics.drop(columns="split").round(3))

,evaluation_protocol,reference,subjects,windows,windows_per_subject_min,windows_per_subject_median,windows_per_subject_max,subject_mean_mae,subject_mean_rmse,pooled_mae,pooled_rmse
0,Unseen-subject Generalization,linear_trend_30min,9,24259,110,2916.0,3228,14.024,20.886,14.751,22.008
1,Unseen-subject Generalization,persistence,9,24259,110,2916.0,3228,11.428,17.210,12.036,18.606
2,Within-subject Future Prediction,linear_trend_30min,45,26330,1,562.0,1073,13.724,19.849,13.298,20.142
3,Within-subject Future Prediction,persistence,45,26330,1,562.0,1073,10.616,15.443,10.429,16.236


Persistence equals the published value at $t$, labels equal the published value at exactly $t+30$, no predictor source time is later than $t$, window supports lie inside the validation partitions, and Protocol B validation subjects are absent from its training split. Windows per subject range from 1 to 1,073 in Protocol A, so subject-equal weighting gives a single-window subject full weight.

# Training-Only Preprocessing and Model Fitting

The two formal baselines selected earlier in this notebook and two small project comparisons are fitted on the training range of each evaluation protocol. Configurations are fixed before fitting: GlucoBench linear regression (OLS for the 30-min step, explicit intercept, target lags at 5-min spacing with `in_len` $\in \{12, 24\}$, min-max scaling fitted on training rows); Sparacino AR(1)-WLS (per-window closed-form exponentially weighted estimate over 41 observations at 3-min spacing, $T = 10$, no filter, $\mu \in \{0.2, 0.5, 0.8\}$); Ridge on the registered glucose-history features ($\alpha \in \{0.1, 1, 10\}$, training-only standardization); and shallow gradient boosting (depth $= 3$, learning rate $= 0.1$, `max_iter` $\in \{100, 300\}$, no internal early stopping). Within each model and protocol, the configuration with the lowest validation subject-mean $\mathrm{RMSE}$ is selected. Selected models are not refitted on validation data, no imputation is required, and test rows are excluded before any glucose value is read.

In [11]:
# Load admitted training and validation rows with frozen glucose-history features.
REGISTERED_GLUCOSE_FEATURES = spec_task["registered_glucose_features"]
FIT_SPLITS = ["train", "validation"]

# Admitted train/validation rows of each protocol; test rows are dropped before any glucose value is read.
folds = canonical_folds[FOLD_KEY + ["split", "admitted_to_split"]].copy()
rows = folds.loc[folds["admitted_to_split"] & folds["split"].isin(FIT_SPLITS)].drop(columns="admitted_to_split")
rows = rows.sort_values(FOLD_KEY).reset_index(drop=True)
allowed_windows = rows[WINDOW_KEY].drop_duplicates().reset_index(drop=True)

# Labels and registered features for the allowed windows only.
samples = canonical_samples[WINDOW_KEY + ["target_value"] + REGISTERED_GLUCOSE_FEATURES].copy()
samples = allowed_windows.merge(samples, on=WINDOW_KEY, how="left", validate="one_to_one")
assert samples[["target_value"] + REGISTERED_GLUCOSE_FEATURES].notna().all().all()

# Glucose values at fixed offsets from the origin, gathered from the frozen contiguous 1-minute grid.
LAG_OFFSETS = sorted(set(range(-HISTORY_MIN, 1, 5)) | set(range(-HISTORY_MIN, 1, 3)))
grid = base_grid.loc[base_grid["subject"].isin(allowed_windows["subject"].unique()),
                     ["subject", "timestamp", TARGET_FIELD]].sort_values(["subject", "timestamp"])
assert grid.groupby("subject")["timestamp"].diff().dropna().eq(pd.Timedelta(minutes=1)).all()
subject_start = grid.groupby("subject")["timestamp"].first()
subject_values = {s: part[TARGET_FIELD].to_numpy() for s, part in grid.groupby("subject")}
del grid
origin_index = ((samples["origin_time"] - samples["subject"].map(subject_start)) // pd.Timedelta(minutes=1)).to_numpy()
lag_matrix = np.empty((len(samples), len(LAG_OFFSETS)))
for s, idx in samples.groupby("subject").indices.items():
    positions = origin_index[idx][:, None] + np.array(LAG_OFFSETS)[None, :]
    lag_matrix[idx] = subject_values[s][positions]
del subject_values
assert np.isfinite(lag_matrix).all() and max(LAG_OFFSETS) == 0
lags = pd.DataFrame(lag_matrix, columns=[f"g{o}" for o in LAG_OFFSETS])

# Cross-check gathered values against the registered exact-lag features.
for minutes in [0, 30, 60, 120]:
    assert np.allclose(lags[f"g{-minutes}"], samples[f"glucose_lag_{minutes}min"], rtol=0, atol=1e-9)

windows = pd.concat([samples, lags], axis=1)
data = rows.merge(windows, on=WINDOW_KEY, how="left", validate="many_to_one")
display(data.groupby(["evaluation_protocol", "split"]).agg(windows=("subject", "size"), subjects=("subject", "nunique")))

windows  subjects
evaluation_protocol              split                        
Unseen-subject Generalization    train         81958        27
                                 validation    24259         9
Within-subject Future Prediction train         79300        45
                                 validation    26330        45

In [12]:
# Define fitting and evaluation helpers for the frozen model configurations.
def evaluate(frame: pd.DataFrame, prediction: np.ndarray) -> dict:
    """Subject-equal-weight (primary) and pooled-window MAE/RMSE."""
    errors = prediction - frame["target_value"].to_numpy()
    per_subject = pd.DataFrame([error_summary(errors[idx]) for idx in frame.groupby("subject").indices.values()])
    pooled = error_summary(errors)
    return {"subjects": len(per_subject), "windows": pooled["windows"],
            "subject_mean_mae": per_subject["mae"].mean(), "subject_mean_rmse": per_subject["rmse"].mean(),
            "pooled_mae": pooled["mae"], "pooled_rmse": pooled["rmse"]}


def subject_metrics(frame: pd.DataFrame, prediction: np.ndarray) -> pd.DataFrame:
    errors = prediction - frame["target_value"].to_numpy()
    return pd.DataFrame([{"subject": s, **error_summary(errors[idx])}
                         for s, idx in frame.groupby("subject").indices.items()])


def lag_columns(step: int, points: int) -> list:
    """Most recent `points` observations at `step`-minute spacing, oldest first, ending at the origin."""
    offsets = list(range(-(points - 1) * step, 1, step))
    assert min(offsets) >= -HISTORY_MIN
    return [f"g{o}" for o in offsets]


def sparacino_coefficient(u: np.ndarray, mu: float) -> np.ndarray:
    """Closed-form exponentially weighted LS for u_i = a u_(i-1); u has shape (n_windows, n_obs), oldest first."""
    x, y = u[:, :-1], u[:, 1:]
    weights = mu ** np.arange(x.shape[1] - 1, -1, -1)
    return (weights * x * y).sum(axis=1) / (weights * x * x).sum(axis=1)


def sparacino_forecast(u: np.ndarray, mu: float, steps: int) -> np.ndarray:
    return sparacino_coefficient(u, mu) ** steps * u[:, -1]


def rls_coefficient(u_row: np.ndarray, mu: float, p0: float = 1e12) -> float:
    """Independent scalar recursive least squares with forgetting factor mu, processed in time order."""
    theta, p = 0.0, p0
    for x_prev, x_now in zip(u_row[:-1], u_row[1:]):
        gain = p * x_prev / (mu + x_prev * p * x_prev)
        theta += gain * (x_now - theta * x_prev)
        p = p / (mu + x_prev * p * x_prev)  # scalar form of (p - gain * x_prev * p) / mu, avoids cancellation
    return theta


SPARACINO_STEP = frozen_spec["baselines"]["sparacino2007_ar1_wls"]["geometry"]["step_minutes"]
SPARACINO_T = frozen_spec["baselines"]["sparacino2007_ar1_wls"]["geometry"]["forecast_steps_T"]
SPARACINO_COLUMNS = [f"g{o}" for o in frozen_spec["baselines"]["sparacino2007_ar1_wls"]["geometry"]["observation_offsets_minutes"]]
assert len(SPARACINO_COLUMNS) == len(range(-HISTORY_MIN, 1, SPARACINO_STEP)) and SPARACINO_T * SPARACINO_STEP == HORIZON_MIN

In [13]:
# GlucoBench linear regression: one OLS model for the 30-min step on target lags at 5-min spacing, per protocol.
LR_CANDIDATES = [12, 24]
fits, metric_rows, subject_rows, selections = {}, [], [], {}


def record(model_id, protocol, config, split, frame, prediction, runtime=None):
    metric_rows.append({"model_id": model_id, "evaluation_protocol": protocol, "config": config, "split": split,
                        **evaluate(frame, prediction), "fit_seconds": runtime})


for protocol in (PROTOCOL_A, PROTOCOL_B):
    train = data.loc[data["evaluation_protocol"].eq(protocol) & data["split"].eq("train")]
    valid = data.loc[data["evaluation_protocol"].eq(protocol) & data["split"].eq("validation")]
    for in_len in LR_CANDIDATES:
        columns = lag_columns(5, in_len)
        start = time.perf_counter()
        # Training-only min-max scaling of the glucose signal (lags and target share one scaler).
        scaler = MinMaxScaler().fit(np.r_[train[columns].to_numpy().ravel(), train["target_value"].to_numpy()].reshape(-1, 1))
        scale = lambda frame: scaler.transform(frame[columns].to_numpy().reshape(-1, 1)).reshape(-1, len(columns))
        model = LinearRegression(fit_intercept=True).fit(scale(train), scaler.transform(train[["target_value"]].to_numpy()).ravel())
        runtime = time.perf_counter() - start
        predict = lambda frame: scaler.inverse_transform(model.predict(scale(frame)).reshape(-1, 1)).ravel()
        config = f"in_len={in_len}"
        record("glucobench_linear_regression", protocol, config, "train", train, predict(train), runtime)
        record("glucobench_linear_regression", protocol, config, "validation", valid, predict(valid))
        fits[("glucobench_linear_regression", protocol, config)] = {
            "model": model, "scaler": scaler, "lag_columns": columns,
            "params": {"in_len": in_len, "lag_step_minutes": 5, "fit_intercept": True,
                       "intercept_scaled": float(model.intercept_), "coef_scaled": model.coef_.tolist(),
                       "scaler_data_min": float(scaler.data_min_[0]), "scaler_data_max": float(scaler.data_max_[0]),
                       "training_windows": len(train)}}

lr_metrics = pd.DataFrame(metric_rows)
display(lr_metrics.round(3))

,model_id,evaluation_protocol,config,split,subjects,windows,subject_mean_mae,subject_mean_rmse,pooled_mae,pooled_rmse,fit_seconds
0,glucobench_linear_regression,Within-subject Future Prediction,in_len=12,train,45,79300,8.129,11.854,8.094,11.974,0.141
1,glucobench_linear_regression,Within-subject Future Prediction,in_len=12,validation,45,26330,7.699,11.223,7.600,11.588,NaN
2,glucobench_linear_regression,Within-subject Future Prediction,in_len=24,train,45,79300,8.020,11.706,7.981,11.816,0.215
3,glucobench_linear_regression,Within-subject Future Prediction,in_len=24,validation,45,26330,7.567,11.075,7.481,11.485,NaN
4,glucobench_linear_regression,Unseen-subject Generalization,in_len=12,train,27,81958,7.834,11.427,7.843,11.605,0.102
5,glucobench_linear_regression,Unseen-subject Generalization,in_len=12,validation,9,24259,8.449,12.492,8.299,12.538,NaN
6,glucobench_linear_regression,Unseen-subject Generalization,in_len=24,train,27,81958,7.719,11.258,7.728,11.435,0.213
7,glucobench_linear_regression,Unseen-subject Generalization,in_len=24,validation,9,24259,8.342,12.379,8.186,12.378,NaN


In [14]:
# Sparacino AR(1)-WLS: local coefficient per window, no fitting; mu chosen on validation within each protocol (R2).
MU_CANDIDATES = [0.2, 0.5, 0.8]
for protocol in (PROTOCOL_A, PROTOCOL_B):
    for split in FIT_SPLITS:
        frame = data.loc[data["evaluation_protocol"].eq(protocol) & data["split"].eq(split)]
        u = frame[SPARACINO_COLUMNS].to_numpy()
        for mu in MU_CANDIDATES:
            start = time.perf_counter()
            prediction = sparacino_forecast(u, mu, SPARACINO_T)
            runtime = time.perf_counter() - start
            record("sparacino2007_ar1_wls", protocol, f"mu={mu}", split, frame, prediction, runtime)
            fits[("sparacino2007_ar1_wls", protocol, f"mu={mu}")] = {
                "model": None, "scaler": None, "lag_columns": SPARACINO_COLUMNS,
                "params": {"mu": mu, "step_minutes": SPARACINO_STEP, "T": SPARACINO_T, "intercept": False,
                           "filter": "none (R1)", "estimator": "closed-form exponentially weighted LS (R3)",
                           "fitted_parameters": "none; coefficient a is computed per window"}}

sparacino_metrics = pd.DataFrame(metric_rows).query("model_id == 'sparacino2007_ar1_wls'")
display(sparacino_metrics.round(3))

,model_id,evaluation_protocol,config,split,subjects,windows,subject_mean_mae,subject_mean_rmse,pooled_mae,pooled_rmse,fit_seconds
8,sparacino2007_ar1_wls,Within-subject Future Prediction,mu=0.2,train,45,79300,11.585,18.066,11.633,18.265,0.093
9,sparacino2007_ar1_wls,Within-subject Future Prediction,mu=0.5,train,45,79300,11.884,18.435,11.931,18.648,0.102
10,sparacino2007_ar1_wls,Within-subject Future Prediction,mu=0.8,train,45,79300,12.464,19.126,12.525,19.458,0.096
11,sparacino2007_ar1_wls,Within-subject Future Prediction,mu=0.2,validation,45,26330,11.191,17.518,10.762,17.994,0.028
12,sparacino2007_ar1_wls,Within-subject Future Prediction,mu=0.5,validation,45,26330,11.486,17.884,11.075,18.351,0.030
13,sparacino2007_ar1_wls,Within-subject Future Prediction,mu=0.8,validation,45,26330,12.108,18.600,11.759,19.084,0.032
14,sparacino2007_ar1_wls,Unseen-subject Generalization,mu=0.2,train,27,81958,11.357,17.448,11.383,17.741,0.101
15,sparacino2007_ar1_wls,Unseen-subject Generalization,mu=0.5,train,27,81958,11.630,17.788,11.658,18.090,0.104
16,sparacino2007_ar1_wls,Unseen-subject Generalization,mu=0.8,train,27,81958,12.148,18.408,12.182,18.749,0.105
17,sparacino2007_ar1_wls,Unseen-subject Generalization,mu=0.2,validation,9,24259,11.320,18.217,11.677,18.524,0.036


In [15]:
# Project comparisons (not reproductions): Ridge and shallow gradient boosting on the registered glucose-history features.
COMPARISON_GRID = {
    "ridge_glucose_history_features": [{"alpha": a} for a in (0.1, 1.0, 10.0)],
    "shallow_gbt_glucose_history_features": [{"max_iter": m, "max_depth": 3, "learning_rate": 0.1} for m in (100, 300)],
}
for protocol in (PROTOCOL_A, PROTOCOL_B):
    train = data.loc[data["evaluation_protocol"].eq(protocol) & data["split"].eq("train")]
    valid = data.loc[data["evaluation_protocol"].eq(protocol) & data["split"].eq("validation")]
    for model_id, grid_configs in COMPARISON_GRID.items():
        for params in grid_configs:
            config = ",".join(f"{k}={v}" for k, v in params.items())
            start = time.perf_counter()
            if model_id.startswith("ridge"):
                scaler = StandardScaler().fit(train[REGISTERED_GLUCOSE_FEATURES])  # training rows only
                model = Ridge(**params).fit(scaler.transform(train[REGISTERED_GLUCOSE_FEATURES]), train["target_value"])
                predict = lambda frame: model.predict(scaler.transform(frame[REGISTERED_GLUCOSE_FEATURES]))
                state = {"scaler_mean": scaler.mean_.tolist(), "scaler_scale": scaler.scale_.tolist(),
                         "intercept": float(model.intercept_), "coef": model.coef_.tolist()}
            else:
                scaler = None
                model = HistGradientBoostingRegressor(**params, early_stopping=False, random_state=0).fit(
                    train[REGISTERED_GLUCOSE_FEATURES], train["target_value"])
                predict = lambda frame: model.predict(frame[REGISTERED_GLUCOSE_FEATURES])
                state = {"n_iter": int(model.n_iter_)}
            runtime = time.perf_counter() - start
            record(model_id, protocol, config, "train", train, predict(train), runtime)
            record(model_id, protocol, config, "validation", valid, predict(valid))
            fits[(model_id, protocol, config)] = {"model": model, "scaler": scaler, "lag_columns": REGISTERED_GLUCOSE_FEATURES,
                                                  "params": {**params, **state, "training_windows": len(train)}}

comparison_metrics = pd.DataFrame(metric_rows).loc[lambda d: d["model_id"].isin(list(COMPARISON_GRID))]
display(comparison_metrics.round(3))

,model_id,evaluation_protocol,config,split,subjects,windows,subject_mean_mae,subject_mean_rmse,pooled_mae,pooled_rmse,fit_seconds
20,ridge_glucose_history_features,Within-subject Future Prediction,alpha=0.1,train,45,79300,8.295,12.046,8.278,12.185,0.075
21,ridge_glucose_history_features,Within-subject Future Prediction,alpha=0.1,validation,45,26330,7.974,11.477,7.767,11.721,NaN
22,ridge_glucose_history_features,Within-subject Future Prediction,alpha=1.0,train,45,79300,8.284,12.047,8.268,12.191,0.078
23,ridge_glucose_history_features,Within-subject Future Prediction,alpha=1.0,validation,45,26330,7.971,11.485,7.759,11.722,NaN
24,ridge_glucose_history_features,Within-subject Future Prediction,alpha=10.0,train,45,79300,8.332,12.147,8.320,12.323,0.087
25,ridge_glucose_history_features,Within-subject Future Prediction,alpha=10.0,validation,45,26330,8.037,11.592,7.824,11.841,NaN
26,shallow_gbt_glucose_history_features,Within-subject Future Prediction,"max_iter=100,max_depth=3,learning_rate=0.1",train,45,79300,9.778,14.132,9.844,14.546,4.290
27,shallow_gbt_glucose_history_features,Within-subject Future Prediction,"max_iter=100,max_depth=3,learning_rate=0.1",validation,45,26330,9.483,13.599,9.455,14.216,NaN
28,shallow_gbt_glucose_history_features,Within-subject Future Prediction,"max_iter=300,max_depth=3,learning_rate=0.1",train,45,79300,9.250,13.287,9.315,13.658,2.484
29,shallow_gbt_glucose_history_features,Within-subject Future Prediction,"max_iter=300,max_depth=3,learning_rate=0.1",validation,45,26330,9.076,12.972,9.043,13.563,NaN


In [16]:
# Configuration selection per model and protocol: lowest validation subject-mean RMSE (validation only).
all_metrics = pd.DataFrame(metric_rows)
validation_metrics = all_metrics.loc[all_metrics["split"].eq("validation")]
selected = (validation_metrics.sort_values("subject_mean_rmse", kind="stable")
            .groupby(["model_id", "evaluation_protocol"], sort=False).head(1)
            [["model_id", "evaluation_protocol", "config"]])
all_metrics = all_metrics.merge(selected.assign(selected=True), on=["model_id", "evaluation_protocol", "config"],
                                how="left")
all_metrics["selected"] = all_metrics["selected"].eq(True)

# Per-subject metrics for the selected configurations.
for row in selected.itertuples(index=False):
    fit = fits[(row.model_id, row.evaluation_protocol, row.config)]
    for split in FIT_SPLITS:
        frame = data.loc[data["evaluation_protocol"].eq(row.evaluation_protocol) & data["split"].eq(split)]
        if row.model_id == "sparacino2007_ar1_wls":
            prediction = sparacino_forecast(frame[SPARACINO_COLUMNS].to_numpy(), fit["params"]["mu"], SPARACINO_T)
        elif row.model_id == "glucobench_linear_regression":
            columns, scaler = fit["lag_columns"], fit["scaler"]
            scaled = scaler.transform(frame[columns].to_numpy().reshape(-1, 1)).reshape(-1, len(columns))
            prediction = scaler.inverse_transform(fit["model"].predict(scaled).reshape(-1, 1)).ravel()
        elif fit["scaler"] is not None:
            prediction = fit["model"].predict(fit["scaler"].transform(frame[REGISTERED_GLUCOSE_FEATURES]))
        else:
            prediction = fit["model"].predict(frame[REGISTERED_GLUCOSE_FEATURES])
        subject_rows.append(subject_metrics(frame, prediction).assign(
            model_id=row.model_id, evaluation_protocol=row.evaluation_protocol, config=row.config, split=split))

MODEL_ROLES = {"glucobench_linear_regression": "formal baseline (primary)",
               "sparacino2007_ar1_wls": "formal baseline (secondary)",
               "ridge_glucose_history_features": "project comparison", "shallow_gbt_glucose_history_features": "project comparison"}
all_metrics["role"] = all_metrics["model_id"].map(MODEL_ROLES)
display(all_metrics.loc[all_metrics["selected"] & all_metrics["split"].eq("validation"),
                        ["role", "model_id", "evaluation_protocol", "config", "subject_mean_mae",
                         "subject_mean_rmse", "pooled_mae", "pooled_rmse"]].round(3))

,role,model_id,evaluation_protocol,config,subject_mean_mae,subject_mean_rmse,pooled_mae,pooled_rmse
3,formal baseline (primary),glucobench_linear_regression,Within-subject Future Prediction,in_len=24,7.567,11.075,7.481,11.485
7,formal baseline (primary),glucobench_linear_regression,Unseen-subject Generalization,in_len=24,8.342,12.379,8.186,12.378
11,formal baseline (secondary),sparacino2007_ar1_wls,Within-subject Future Prediction,mu=0.2,11.191,17.518,10.762,17.994
17,formal baseline (secondary),sparacino2007_ar1_wls,Unseen-subject Generalization,mu=0.2,11.320,18.217,11.677,18.524
21,project comparison,ridge_glucose_history_features,Within-subject Future Prediction,alpha=0.1,7.974,11.477,7.767,11.721
29,project comparison,shallow_gbt_glucose_history_features,Within-subject Future Prediction,"max_iter=300,max_depth=3,learning_rate=0.1",9.076,12.972,9.043,13.563
31,project comparison,ridge_glucose_history_features,Unseen-subject Generalization,alpha=0.1,8.573,12.682,8.494,12.791
39,project comparison,shallow_gbt_glucose_history_features,Unseen-subject Generalization,"max_iter=300,max_depth=3,learning_rate=0.1",9.957,14.686,10.282,15.471


Selection used validation rows only. The selected LR history (24 lags) is the largest allowed by the frozen window; the selected Sparacino $\mu = 0.2$ and GBT size (300 iterations) lie at the edge of their pre-specified grids, which were not extended.

In [17]:
# Leakage and isolation checks on what was fitted and scored.
isolation = {}
all_folds = canonical_folds[FOLD_KEY + ["split", "admitted_to_split"]]
test_keys = all_folds.loc[all_folds["split"].eq("test"), FOLD_KEY]
isolation["no test (protocol, window) row in fitted/scored data"] = data.merge(test_keys, on=FOLD_KEY).empty
for protocol in (PROTOCOL_A, PROTOCOL_B):
    part = data.loc[data["evaluation_protocol"].eq(protocol)]
    train_keys = set(map(tuple, part.loc[part["split"].eq("train"), WINDOW_KEY].to_numpy()))
    valid_keys = set(map(tuple, part.loc[part["split"].eq("validation"), WINDOW_KEY].to_numpy()))
    isolation[f"{protocol}: train and validation windows disjoint"] = train_keys.isdisjoint(valid_keys)
b_subjects = {split: set(all_folds.loc[all_folds["evaluation_protocol"].eq(PROTOCOL_B) & all_folds["split"].eq(split), "subject"])
              for split in ["train", "validation", "test"]}
isolation["Protocol B training subjects disjoint from validation and test subjects"] = \
    b_subjects["train"].isdisjoint(b_subjects["validation"] | b_subjects["test"])

# Inputs end at the origin; the label lies 30 min later.
input_offsets = [int(c[1:]) for fit in fits.values() for c in fit["lag_columns"] if re.fullmatch(r"g-?\d+", c)]
isolation["all lag inputs at or before the origin"] = max(input_offsets) <= 0
isolation["registered glucose-history features end at the origin"] = bool(
    feature_registry_by_name.loc[REGISTERED_GLUCOSE_FEATURES, "maximum_source_time_relative_to_origin_min"].le(0).all())

isolation_table = pd.Series(isolation, name="passed").to_frame()
assert isolation_table["passed"].all(), isolation_table
display(isolation_table)

,passed
"no test (protocol, window) row in fitted/scored data",True
Within-subject Future Prediction: train and validation windows disjoint,True
Unseen-subject Generalization: train and validation windows disjoint,True
Protocol B training subjects disjoint from validation and test subjects,True
all lag inputs at or before the origin,True
registered glucose-history features end at the origin,True


In [18]:
# Persist selected fits (baseline_fits.joblib holds only the validation-selected configuration per model/protocol) and validation metrics.
fits_path = BASELINE_DIR / "baseline_fits.joblib"
metrics_path = BASELINE_DIR / "baseline_validation_metrics.csv"

# Store fitted objects; fit metadata remains in the validation table and registry.
selected_keys = set(map(tuple, selected[["model_id", "evaluation_protocol", "config"]].to_numpy()))
joblib.dump({"spec_id": frozen_spec["spec_id"],
             "fits": {"|".join(k): {"model": v["model"], "scaler": v["scaler"], "inputs": v["lag_columns"]}
                      for k, v in fits.items() if k in selected_keys}}, fits_path)
fit_records = [{"model_id": k[0], "role": MODEL_ROLES[k[0]], "evaluation_protocol": k[1], "config": k[2],
                "selected": k in selected_keys, "inputs": v["lag_columns"], "params": v["params"],
                "fit_seconds": float(all_metrics.loc[(all_metrics["model_id"].eq(k[0]))
                                                     & all_metrics["evaluation_protocol"].eq(k[1])
                                                     & all_metrics["config"].eq(k[2])
                                                     & all_metrics["split"].eq("train"), "fit_seconds"].iloc[0])}
               for k, v in fits.items()]
fits_record = {
    "spec_id": frozen_spec["spec_id"],
    "selection_rule": "lowest validation subject-mean RMSE within each model and protocol; test never used",
    "final_refit": "none; selected models remain fitted on the training split only",
    "imputation": "none required; all frozen windows are complete",
    "fits": fit_records,
}

metrics_table = pd.concat([
    all_metrics.assign(scope="summary"),
    pd.concat(subject_rows, ignore_index=True).assign(scope="subject", selected=True),
], ignore_index=True)
metrics_table["role"] = metrics_table["model_id"].map(MODEL_ROLES)
metrics_table.to_csv(metrics_path, index=False, encoding="utf-8", lineterminator="\n")
print(f"Candidate fits evaluated: {len(fits)}; selected fits persisted: {len(selected_keys)}; metric rows: {len(metrics_table)}")

Candidate fits evaluated: 20; selected fits persisted: 8; metric rows: 544


Test windows are absent from fitting and scoring, train and validation windows are disjoint within each protocol, Protocol B training subjects are disjoint from its validation and test subjects, and every lag input lies at or before the origin. GlucoBench LR is below the persistence reference in both protocols; the Sparacino baseline is above it.

# Reproduction Audit and Validation Comparison

The implementations of the two formal baselines are audited against their source methods, and all models are compared on the common validation windows in three separate groups: formal baseline reproductions, simple references, and project comparisons. Stored fits are reused without refitting and test data are not read.

In [19]:
# Collect fitted models, validation metrics, and reference predictions.
fitted = {"|".join(key): {"model": value["model"], "scaler": value["scaler"], "inputs": value["lag_columns"]}
          for key, value in fits.items()}
fitted_metrics = metrics_table

# Selected configurations as recorded in the fit record.
selected_fits = pd.DataFrame([{k: r[k] for k in ["model_id", "role", "evaluation_protocol", "config"]}
                              for r in fits_record["fits"] if r["selected"]])

# Validation windows must be identical between the reference predictions and the fitted models.
validation_rows = data.loc[data["split"].eq("validation")]
reference_keys = reference_predictions.loc[reference_predictions["reference"].eq("persistence"), FOLD_KEY + ["target_value"]]
key_check = validation_rows[FOLD_KEY + ["target_value"]].merge(reference_keys, on=FOLD_KEY, how="outer",
                                                               suffixes=("", "_ref"), indicator=True)
assert key_check["_merge"].eq("both").all() and np.allclose(key_check["target_value"], key_check["target_value_ref"])

display(selected_fits)
print(f"Validation windows shared by the reference and model comparisons: {len(key_check)}")

,model_id,role,evaluation_protocol,config
0,glucobench_linear_regression,formal baseline (primary),Within-subject Future Prediction,in_len=24
1,glucobench_linear_regression,formal baseline (primary),Unseen-subject Generalization,in_len=24
2,sparacino2007_ar1_wls,formal baseline (secondary),Within-subject Future Prediction,mu=0.2
3,sparacino2007_ar1_wls,formal baseline (secondary),Unseen-subject Generalization,mu=0.2
4,ridge_glucose_history_features,project comparison,Within-subject Future Prediction,alpha=0.1
5,shallow_gbt_glucose_history_features,project comparison,Within-subject Future Prediction,"max_iter=300,max_depth=3,learning_rate=0.1"
6,ridge_glucose_history_features,project comparison,Unseen-subject Generalization,alpha=0.1
7,shallow_gbt_glucose_history_features,project comparison,Unseen-subject Generalization,"max_iter=300,max_depth=3,learning_rate=0.1"


Validation windows shared by the reference and model comparisons: 50589


In [20]:
# Predict the validation windows with the selected fits (no refitting).
def predict_selected(model_id: str, protocol: str, config: str, frame: pd.DataFrame) -> np.ndarray:
    fit = fitted["|".join([model_id, protocol, config])]
    if model_id == "sparacino2007_ar1_wls":
        mu = next(r["params"]["mu"] for r in fits_record["fits"]
                  if (r["model_id"], r["evaluation_protocol"], r["config"]) == (model_id, protocol, config))
        return sparacino_forecast(frame[fit["inputs"]].to_numpy(), mu, SPARACINO_T)
    if model_id == "glucobench_linear_regression":
        scaler, columns = fit["scaler"], fit["inputs"]
        scaled = scaler.transform(frame[columns].to_numpy().reshape(-1, 1)).reshape(-1, len(columns))
        return scaler.inverse_transform(fit["model"].predict(scaled).reshape(-1, 1)).ravel()
    features = frame[fit["inputs"]]
    return fit["model"].predict(fit["scaler"].transform(features) if fit["scaler"] is not None else features)

prediction_frames = []
for row in selected_fits.itertuples(index=False):
    frame = data.loc[data["evaluation_protocol"].eq(row.evaluation_protocol) & data["split"].eq("validation")]
    prediction = predict_selected(row.model_id, row.evaluation_protocol, row.config, frame)
    prediction_frames.append(frame[FOLD_KEY + ["target_value"]].assign(model_id=row.model_id, prediction=prediction))

# Reference predictions from the persistence/trend section enter the same long table.
for reference in ["persistence", "linear_trend_30min"]:
    part = reference_predictions.loc[reference_predictions["reference"].eq(reference)]
    prediction_frames.append(part[FOLD_KEY + ["target_value", "prediction"]].assign(model_id=reference))
validation_predictions = pd.concat(prediction_frames, ignore_index=True)
validation_predictions["error"] = validation_predictions["prediction"] - validation_predictions["target_value"]

In [21]:
# Source method versus CGMacros implementation, per aspect; values of the implementation come from the fit record.
selected_params = {(r["model_id"], r["evaluation_protocol"]): r["params"] for r in fits_record["fits"] if r["selected"]}
lr_in_len = sorted({p["in_len"] for (m, _), p in selected_params.items() if m == "glucobench_linear_regression"})
sp_mu = sorted({p["mu"] for (m, _), p in selected_params.items() if m == "sparacino2007_ar1_wls"})
lr_intercept = {p["fit_intercept"] for (m, _), p in selected_params.items() if m == "glucobench_linear_regression"}

CHOICE, SAME = "project choice (not reported in source)", "matches source"
# Compact implementation/adaptation table: what CGMacros implements and how it relates to the source method.
# Full source-method citations and provenance are recorded in the sources/baseline_fields tables above and in the report.
method_rows = [
    ("glucobench_linear_regression", "forecast target", f"single direct OLS model for t+{HORIZON_MIN} min (the 30-min step)", ADAPTED),
    ("glucobench_linear_regression", "input length", f"in_len in {{12, 24}} chosen on validation subject-mean RMSE; selected {lr_in_len}", ADAPTED),
    ("glucobench_linear_regression", "sampling of lags", "5-min points drawn from the released 1-min Libre grid; strict windows, no project interpolation", ADAPTED),
    ("glucobench_linear_regression", "estimator", "sklearn LinearRegression fitted directly for the single step", CHOICE),
    ("glucobench_linear_regression", "intercept", f"fit_intercept={sorted(lr_intercept)[0]} set explicitly", CHOICE),
    ("glucobench_linear_regression", "scaling", "one MinMaxScaler per protocol fitted on training lags and targets", SAME),
    ("glucobench_linear_regression", "static covariates", "none; subject identity is a grouping key only", ADAPTED),
    ("glucobench_linear_regression", "training rows", "frozen Notebook 03 training windows of each protocol (origins every 5 min)", ADAPTED),
    ("sparacino2007_ar1_wls", "model", "same equation, no intercept", SAME),
    ("sparacino2007_ar1_wls", "weighting", "same weights over the window pairs", SAME),
    ("sparacino2007_ar1_wls", "estimator", "closed-form exponentially weighted LS, cross-checked against an independent RLS in the verification checks", DECISION),
    ("sparacino2007_ar1_wls", "forgetting factor", f"mu selected on validation within each protocol; selected {sp_mu}", DECISION),
    ("sparacino2007_ar1_wls", "memory", f"frozen window [t-{HISTORY_MIN}, t]: 41 observations at 3 min; no state across windows", ADAPTED),
    ("sparacino2007_ar1_wls", "sampling", "3-min points drawn from the released 1-min Libre grid (interpolated publisher values)", ADAPTED),
    ("sparacino2007_ar1_wls", "prefiltering", "no filtering in the main analysis", DECISION),
    ("sparacino2007_ar1_wls", "prediction", f"a^{SPARACINO_T} u_t for t+{HORIZON_MIN} min", SAME),
]
method_audit = pd.DataFrame(method_rows, columns=["model_id", "aspect", "cgmacros_implementation", "status"])
method_audit["record_type"] = "method_difference"

display(method_audit)

,model_id,aspect,cgmacros_implementation,status,record_type
0,glucobench_linear_regression,forecast target,single direct OLS model for t+30 min (the 30-min step),project adaptation,method_difference
1,glucobench_linear_regression,input length,"in_len in {12, 24} chosen on validation subject-mean RMSE; selected [24]",project adaptation,method_difference
2,glucobench_linear_regression,sampling of lags,"5-min points drawn from the released 1-min Libre grid; strict windows, no project inte...",project adaptation,method_difference
3,glucobench_linear_regression,estimator,sklearn LinearRegression fitted directly for the single step,project choice (not reported in source),method_difference
4,glucobench_linear_regression,intercept,fit_intercept=True set explicitly,project choice (not reported in source),method_difference
5,glucobench_linear_regression,scaling,one MinMaxScaler per protocol fitted on training lags and targets,matches source,method_difference
6,glucobench_linear_regression,static covariates,none; subject identity is a grouping key only,project adaptation,method_difference
7,glucobench_linear_regression,training rows,frozen Notebook 03 training windows of each protocol (origins every 5 min),project adaptation,method_difference
8,sparacino2007_ar1_wls,model,"same equation, no intercept",matches source,method_difference
9,sparacino2007_ar1_wls,weighting,same weights over the window pairs,matches source,method_difference


Every parameter not reported by the source is recorded as an explicit project choice; no library default is presented as part of the original method.

In [22]:
# Summarize common-window validation comparisons by model category.
CATEGORIES = {
    "glucobench_linear_regression": "formal baseline reproduction",
    "sparacino2007_ar1_wls": "formal baseline reproduction",
    "persistence": "simple reference",
    "linear_trend_30min": "simple reference",
    "ridge_glucose_history_features": "project comparison",
    "shallow_gbt_glucose_history_features": "project comparison",
}
configs = {(r.model_id, r.evaluation_protocol): r.config for r in selected_fits.itertuples(index=False)}

# Per-subject validation errors for every model on the common validation windows.
subject_table = (validation_predictions.groupby(["model_id", "evaluation_protocol", "subject"])["error"]
                 .apply(lambda e: pd.Series(error_summary(e.to_numpy()))).unstack().reset_index())
persistence_rmse = subject_table.loc[subject_table["model_id"].eq("persistence"),
                                     ["evaluation_protocol", "subject", "rmse"]].rename(columns={"rmse": "persistence_rmse"})
subject_table = subject_table.merge(persistence_rmse, on=["evaluation_protocol", "subject"], how="left")

summary_rows = []
for (model_id, protocol), part in validation_predictions.groupby(["model_id", "evaluation_protocol"]):
    subjects = subject_table.loc[subject_table["model_id"].eq(model_id) & subject_table["evaluation_protocol"].eq(protocol)]
    metrics = evaluate(part, part["prediction"].to_numpy())
    train_row = fitted_metrics.loc[fitted_metrics["scope"].eq("summary") & fitted_metrics["model_id"].eq(model_id)
                                   & fitted_metrics["evaluation_protocol"].eq(protocol) & fitted_metrics["split"].eq("train")
                                   & fitted_metrics["config"].eq(configs.get((model_id, protocol)))]
    train_rmse = float(train_row["subject_mean_rmse"].iloc[0]) if len(train_row) else np.nan
    summary_rows.append({
        "category": CATEGORIES[model_id], "model_id": model_id, "evaluation_protocol": protocol,
        "config": configs.get((model_id, protocol), "no parameters"), **metrics,
        "train_subject_mean_rmse": train_rmse,
        "validation_minus_train_rmse": metrics["subject_mean_rmse"] - train_rmse,
        "delta_rmse_vs_persistence": metrics["subject_mean_rmse"] - subjects["persistence_rmse"].mean(),
        "subjects_better_than_persistence": int((subjects["rmse"] < subjects["persistence_rmse"]).sum()),
    })
comparison = pd.DataFrame(summary_rows)
comparison["category"] = pd.Categorical(comparison["category"], list(dict.fromkeys(CATEGORIES.values())))
comparison = comparison.sort_values(["evaluation_protocol", "category", "subject_mean_rmse"]).reset_index(drop=True)
assert comparison.groupby("evaluation_protocol")["windows"].nunique().eq(1).all()

display(comparison[["evaluation_protocol", "category", "model_id", "config", "subject_mean_mae", "subject_mean_rmse",
                    "pooled_rmse", "validation_minus_train_rmse", "delta_rmse_vs_persistence",
                    "subjects_better_than_persistence", "subjects"]].round(3))

,evaluation_protocol,category,model_id,config,subject_mean_mae,subject_mean_rmse,pooled_rmse,validation_minus_train_rmse,delta_rmse_vs_persistence,subjects_better_than_persistence,subjects
0,Unseen-subject Generalization,formal baseline reproduction,glucobench_linear_regression,in_len=24,8.342,12.379,12.378,1.121,-4.831,8,9
1,Unseen-subject Generalization,formal baseline reproduction,sparacino2007_ar1_wls,mu=0.2,11.320,18.217,18.524,0.769,1.007,4,9
2,Unseen-subject Generalization,simple reference,persistence,no parameters,11.428,17.210,18.606,NaN,0.000,0,9
3,Unseen-subject Generalization,simple reference,linear_trend_30min,no parameters,14.024,20.886,22.008,NaN,3.677,0,9
4,Unseen-subject Generalization,project comparison,ridge_glucose_history_features,alpha=0.1,8.573,12.682,12.791,1.046,-4.528,8,9
5,Unseen-subject Generalization,project comparison,shallow_gbt_glucose_history_features,"max_iter=300,max_depth=3,learning_rate=0.1",9.957,14.686,15.471,1.858,-2.524,9,9
6,Within-subject Future Prediction,formal baseline reproduction,glucobench_linear_regression,in_len=24,7.567,11.075,11.485,-0.630,-4.367,42,45
7,Within-subject Future Prediction,formal baseline reproduction,sparacino2007_ar1_wls,mu=0.2,11.191,17.518,17.994,-0.548,2.075,16,45
8,Within-subject Future Prediction,simple reference,persistence,no parameters,10.616,15.443,16.236,NaN,0.000,0,45
9,Within-subject Future Prediction,simple reference,linear_trend_30min,no parameters,13.724,19.849,20.142,NaN,4.407,3,45


Detailed per-subject and stratified error analysis is presented in Notebook 06.

In [23]:
# Isolation checks for the model comparison.
scoring_isolation = {
    "only validation windows scored": set(validation_predictions.merge(data[FOLD_KEY + ["split"]], on=FOLD_KEY)["split"]) == {"validation"},
    "no test key among scored windows": validation_predictions.merge(
        all_folds.loc[all_folds["split"].eq("test"), FOLD_KEY], on=FOLD_KEY).empty,
    "validation windows identical between the references section and the fitting section": bool(key_check["_merge"].eq("both").all()),
}
assert all(scoring_isolation.values())

# validation_comparison: model summary rows and per-subject rows.
validation_comparison = pd.concat([
    comparison.assign(scope="summary", category=comparison["category"].astype(str)),
    subject_table.assign(scope="subject"),
], ignore_index=True)

comparison_path = BASELINE_DIR / "validation_comparison.csv"
validation_comparison.to_csv(comparison_path, index=False, encoding="utf-8", lineterminator="\n")

## Summary

The reference and fitted models are evaluated on the same validation windows and labels (see the comparison table above for exact subject-mean MAE/RMSE by protocol). GlucoBench LR, Ridge, and shallow GBT all improve on the persistence reference in both protocols; the Sparacino AR(1)-WLS baseline and the linear-trend reference are both worse than persistence.

# Baseline Registry


`baseline_registry` is the single entry point for the downstream notebooks. No model, hyperparameter, preprocessing or protocol is re-selected here, and test data are not read.

In [24]:
# Baseline registry for the downstream notebooks, assembled from the specification, fit record and comparison table above (all in memory).
comparison_summary = validation_comparison.loc[validation_comparison["scope"].eq("summary")]
upstream_paths = {**frozen_spec["upstream_artifacts"], "base_grid": frozen_spec["frozen_task"]["base_grid_artifact"]}

# Sample keys and split sizes per protocol (test counts are key counts only; no test value is read).
key_counts = (all_folds.query("admitted_to_split").groupby(["evaluation_protocol", "split"])
              .agg(windows=("subject", "size"), subjects=("subject", "nunique")).reset_index())

# Model entries grouped by category; selected configurations and training-only preprocessing come from the fit record.
REFERENCE_DEFINITIONS = {
    "persistence": "prediction = glucose_lag_0min (published value at the origin); no parameters",
    "linear_trend_30min": f"prediction = glucose_lag_0min + {frozen_spec['frozen_task']['forecast_horizon_minutes']} x glucose_slope_30min; no parameters",
}
REPRODUCTION = CATEGORIES["glucobench_linear_regression"]
models = {category: {} for category in dict.fromkeys(CATEGORIES.values())}
for record in (r for r in fits_record["fits"] if r["selected"]):
    key = "|".join([record["model_id"], record["evaluation_protocol"], record["config"]])
    entry = models[CATEGORIES[record["model_id"]]].setdefault(record["model_id"], {"role": record["role"], "protocols": {}})
    entry["protocols"][record["evaluation_protocol"]] = {
        "config": record["config"], "inputs": record["inputs"], "params": record["params"], "fit_key": key,
        "preprocessing": type(fitted[key]["scaler"]).__name__ if fitted[key]["scaler"] is not None else "none",
        "preprocessing_fitted_on": "training split of this protocol" if fitted[key]["scaler"] is not None else "not applicable",
    }
for model_id, baseline in frozen_spec["baselines"].items():
    models[REPRODUCTION][model_id]["reproduction_status"] = baseline["reproduction_status"]
models[REPRODUCTION]["sparacino2007_ar1_wls"]["decisions"] = frozen_spec["baselines"]["sparacino2007_ar1_wls"]["decisions"]
for reference, definition in REFERENCE_DEFINITIONS.items():
    models["simple reference"][reference] = {"role": "simple reference (not a reproduction)", "definition": definition}
assert set(models[REPRODUCTION]) == set(frozen_spec["baselines"])

# Tuning grids of the project comparisons: every fitted configuration (not only the selected one), keyed by model ID.
GRID_PARAMETERS = ("alpha", "max_iter", "max_depth", "learning_rate")
tuning_grids = {}
for model_id in models["project comparison"]:
    grid = [dict(p) for p in {tuple(sorted((k, v) for k, v in r["params"].items() if k in GRID_PARAMETERS))
                              for r in fits_record["fits"] if r["model_id"] == model_id}]
    tuning_grids[model_id] = sorted(grid, key=lambda p: tuple(p.values()))

# Random seeds used anywhere in the Notebook 04 pipeline.
gbt_seeds = sorted({fitted[e["fit_key"]]["model"].get_params()["random_state"]
                    for e in models["project comparison"]["shallow_gbt_glucose_history_features"]["protocols"].values()})
random_seeds = {
    "shallow_gbt_random_state": gbt_seeds,
    "reference_slope_audit_sample": "DataFrame.sample(random_state=0)",
    "verification_synthetic_series": "numpy.random.default_rng(0)",
    "verification_window_sample": "DataFrame.sample(random_state=0)",
    "other steps": "deterministic (closed-form or ordinary least squares)",
}

registry = {
    "registry_id": "baseline_registry_v1",
    "spec_id": frozen_spec["spec_id"],
    "status": "selected on validation data only; test data not read before Notebook 06",
    "task": {k: frozen_spec["frozen_task"][k] for k in ["task_id", "target_field", "target_unit", "history_interval",
                                                        "forecast_horizon_minutes", "origin_stride_minutes", "grid_resolution_minutes"]},
    "protocols": frozen_spec["protocol_rules"],
    "protocol_names": {"A": PROTOCOL_A, "B": PROTOCOL_B},
    "sample_keys": {"window_key": frozen_spec["frozen_task"]["window_key"], "fold_key": frozen_spec["frozen_task"]["fold_key"],
                    "admitted_counts": key_counts.to_dict(orient="records")},
    "upstream_paths": upstream_paths,
    "random_seeds": random_seeds,
    "selection_rule": fits_record["selection_rule"],
    "final_refit": fits_record["final_refit"],
    "imputation": fits_record["imputation"],
    "models": models,
    "tuning_grids": tuning_grids,
    "validation_results": comparison_summary[
        ["category", "model_id", "evaluation_protocol", "config", "subjects", "windows", "subject_mean_mae",
         "subject_mean_rmse", "pooled_mae", "pooled_rmse"]].to_dict(orient="records"),
}
display(pd.DataFrame([{"category": c, "model_id": m, "protocols": len(v.get("protocols", {})) or "-"}
                      for c, group in models.items() for m, v in group.items()]))
display(key_counts)

,category,model_id,protocols
0,formal baseline reproduction,glucobench_linear_regression,2
1,formal baseline reproduction,sparacino2007_ar1_wls,2
2,simple reference,persistence,-
3,simple reference,linear_trend_30min,-
4,project comparison,ridge_glucose_history_features,2
5,project comparison,shallow_gbt_glucose_history_features,2


,evaluation_protocol,split,windows,subjects
0,Unseen-subject Generalization,test,28322,9
1,Unseen-subject Generalization,train,81958,27
2,Unseen-subject Generalization,validation,24259,9
3,Within-subject Future Prediction,test,26269,45
4,Within-subject Future Prediction,train,79300,45
5,Within-subject Future Prediction,validation,26330,45


In [25]:
# Persist the baseline registry.
registry_path = BASELINE_DIR / "baseline_registry.json"
registry_path.write_text(json.dumps(registry, indent=2, default=str) + "\n", encoding="utf-8")
print(f"Baseline registry {registry['registry_id']} for spec {registry['spec_id']}: "
      f"{sum(len(group) for group in models.values())} models in {len(models)} categories")

Baseline registry baseline_registry_v1 for spec baseline_reproduction_spec_v2: 6 models in 3 categories
